# UC05 — Detección de Fugas en Red de Distribución

**Objetivo:** Scoring de probabilidad de fuga por DMA (District Metered Area) para priorizar campañas de detección y reducir agua no registrada.

**Tecnologías:** ML.CLASSIFICATION · Streamlit

**Tiempo estimado:** 14 minutos

## Paso 1 — Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS DETECCION_FUGAS_DB;

In [ ]:
USE DATABASE DETECCION_FUGAS_DB;

In [ ]:
USE SCHEMA PUBLIC;

In [ ]:
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;

In [ ]:
USE WAREHOUSE COMPUTE_WH;

## Paso 2 — Generar DMAs y Características de Red

200 zonas de medición con datos de infraestructura hidráulica.

In [ ]:
CREATE OR REPLACE TABLE dmas AS
SELECT
    'DMA-' || LPAD(SEQ4()::STRING, 4, '0') AS dma_id,
    'Zona_' || CASE MOD(SEQ4(), 8) WHEN 0 THEN 'Centro' WHEN 1 THEN 'Norte' WHEN 2 THEN 'Sur' WHEN 3 THEN 'Este' WHEN 4 THEN 'Oeste' WHEN 5 THEN 'Industrial' WHEN 6 THEN 'Residencial' ELSE 'Comercial' END || '_' || SEQ4() AS nombre_zona,
    (50 + UNIFORM(0, 1950, RANDOM()))::INT AS num_acometidas,
    ROUND(2 + UNIFORM(0, 28, RANDOM()), 1) AS longitud_red_km,
    (5 + UNIFORM(0, 55, RANDOM()))::INT AS antiguedad_media_anos,
    CASE MOD(SEQ4(), 4) WHEN 0 THEN 'Fundicion' WHEN 1 THEN 'PVC' WHEN 2 THEN 'Polietileno' ELSE 'Fibrocemento' END AS material_predominante,
    (80 + UNIFORM(0, 420, RANDOM()))::INT AS diametro_medio_mm,
    ROUND(2 + UNIFORM(0, 40, RANDOM()) * 0.1, 1) AS presion_media_bar,
    CASE WHEN UNIFORM(0, 100, RANDOM()) < 35 THEN TRUE ELSE FALSE END AS tiene_regulacion_presion,
    ROUND(CASE WHEN UNIFORM(0, 100, RANDOM()) < 30 THEN 25 + UNIFORM(0, 25, RANDOM()) ELSE 5 + UNIFORM(0, 20, RANDOM()) END, 1) AS anr_pct,
    CASE WHEN UNIFORM(0, 100, RANDOM()) < 30 THEN TRUE ELSE FALSE END AS tiene_fuga_probable
FROM TABLE(GENERATOR(ROWCOUNT => 200));

In [ ]:
SELECT material_predominante, COUNT(*) AS dmas, ROUND(AVG(anr_pct),1) AS anr_medio FROM dmas GROUP BY 1;

## Paso 3 — Feature Engineering y Entrenamiento

In [ ]:
CREATE OR REPLACE TABLE features_fugas AS
SELECT
    anr_pct::FLOAT, presion_media_bar::FLOAT, antiguedad_media_anos::FLOAT,
    CASE WHEN material_predominante = 'Fibrocemento' THEN 1 ELSE 0 END::FLOAT AS es_fibrocemento,
    num_acometidas::FLOAT, longitud_red_km::FLOAT,
    CASE WHEN tiene_regulacion_presion THEN 1 ELSE 0 END::FLOAT AS tiene_regulacion,
    tiene_fuga_probable
FROM dmas;

In [ ]:
CREATE OR REPLACE TABLE train_fugas AS SELECT * FROM features_fugas SAMPLE(80);

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION detector_fugas(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'train_fugas'),
    TARGET_COLNAME => 'tiene_fuga_probable',
    CONFIG_OBJECT => {'evaluate': TRUE}
);

## Paso 4 — Puntuar y Priorizar DMAs

In [ ]:
CREATE OR REPLACE TABLE scoring_fugas AS
SELECT d.*,
    detector_fugas!PREDICT(OBJECT_CONSTRUCT(
        'anr_pct', anr_pct, 'presion_media_bar', presion_media_bar,
        'antiguedad_media_anos', antiguedad_media_anos,
        'es_fibrocemento', CASE WHEN material_predominante='Fibrocemento' THEN 1 ELSE 0 END,
        'num_acometidas', num_acometidas, 'longitud_red_km', longitud_red_km,
        'tiene_regulacion', CASE WHEN tiene_regulacion_presion THEN 1 ELSE 0 END
    )) AS prediccion
FROM dmas d;

In [ ]:
SELECT dma_id, nombre_zona, anr_pct, prediccion:"probability"::OBJECT AS probabilidades
FROM scoring_fugas ORDER BY anr_pct DESC LIMIT 20;

## Paso 5 — Dashboard de Gestión de Fugas

In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session
session = get_active_session()

st.title("🔍 Detección de Fugas — Red de Distribución")

col1, col2, col3 = st.columns(3)
stats = session.sql("SELECT COUNT(*) AS total, ROUND(AVG(anr_pct),1) AS anr_medio, COUNT(CASE WHEN tiene_fuga_probable THEN 1 END) AS con_fuga FROM dmas").collect()[0]
col1.metric("DMAs Monitorizados", stats["TOTAL"])
col2.metric("ANR Medio", f"{stats['ANR_MEDIO']}%")
col3.metric("DMAs con Fuga Probable", stats["CON_FUGA"])

st.subheader("DMAs con Mayor ANR")
df = session.sql("SELECT dma_id, nombre_zona, anr_pct, material_predominante, antiguedad_media_anos FROM dmas ORDER BY anr_pct DESC LIMIT 20").to_pandas()
st.dataframe(df, use_container_width=True)

## Paso 6 — Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS DETECCION_FUGAS_DB;